## UAV Intrusion Detection Stacked Ensemble
This notebook implements a meta-classifier ensemble using One-Class SVM, Autoencoder, and Local Outlier Factor as described in the Whelan et al. (2020) paper into the UAV Attack Dataset by Jason Whelan, et al.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os
import time

### Load the Dataset

In [2]:
# Constants
RANDOM_STATE = 1082
FILE_PATH: str = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "StackedEnsemble_RF"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"
df = pd.read_csv(FILE_PATH)

### Preprocessing

In [3]:
# Prepare features and labels
X = df.drop(columns=["label", "timestamp"])
y = df["label"].map({"benign": 0, "malicious": 1})  # encode labels

# Use benign data to train unsupervised models
X_train_raw = X[y == 0]
X_test_raw = X
y_test = y

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

### Train Base Models

In [4]:
# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_train_pca_svm_lof)
ocsvm_pred = np.where(ocsvm.predict(X_test_pca_svm_lof) == -1, 1, 0)

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_train_pca_svm_lof)
lof_pred = np.where(lof.predict(X_test_pca_svm_lof) == -1, 1, 0)

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
ae.fit(X_train_pca_ae, X_train_pca_ae)

# Compute reconstruction error
recon = ae.predict(X_test_pca_ae)
recon_error = np.mean((X_test_pca_ae - recon) ** 2, axis=1)

# Threshold based on 97.2% percentile from training reconstruction error
train_recon = ae.predict(X_train_pca_ae)
train_error = np.mean((X_train_pca_ae - train_recon) ** 2, axis=1)
threshold = np.percentile(train_error, 97.2)
ae_pred = np.where(recon_error > threshold, 1, 0)

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


### Meta-Classifier Ensemble

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

# Stack predictions from the 3 models
stacked_features = np.vstack([ocsvm_pred, lof_pred, ae_pred]).T

# Meta-classifier (Random Forest)
meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
start_time = time.time()
meta_clf.fit(stacked_features, y_test)
train_time = time.time() - start_time
stacked_pred = meta_clf.predict(stacked_features)

# Evaluation
report = classification_report(
    y_test,
    stacked_pred,
    labels=[0, 1],  # 0 = benign, 1 = malicious
    target_names=["benign", "malicious"],
    zero_division=0,
    output_dict=True
)
f1 = report["weighted avg"]["f1-score"]
accuracy = accuracy_score(y_test, stacked_pred)
print(classification_report(
    y_test,
    stacked_pred,
    labels=[0, 1],
    target_names=["benign", "malicious"],
    zero_division=0
))
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

      benign       1.00      0.99      1.00      3124
   malicious       0.95      1.00      0.97       498

    accuracy                           0.99      3622
   macro avg       0.97      0.99      0.98      3622
weighted avg       0.99      0.99      0.99      3622

F1 Score: 0.9926
Accuracy: 0.9925


In [6]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(meta_clf.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../UAVATTACK_Results.csv
